# Notebook 08 -- Quantum Phase Estimation

**Prerequisites:** Notebooks 01-07. Familiarity with the QFT and phase kickback.

**Quantum Phase Estimation (QPE)** is the most important subroutine in
quantum computing. It extracts the eigenvalue phase of a unitary operator
and underpins Shor's algorithm, quantum chemistry, and the HHL algorithm.

By the end of this notebook you will be able to:

1. **Describe** QPE and predict the phase it extracts from a known unitary.
2. **Implement** QPE using Goqu's `qpe` package.
3. **Use** state preparation to initialize arbitrary quantum states.
4. **Compare** QPE precision with different numbers of phase bits.

For textbook oracle algorithms (Deutsch-Jozsa, Bernstein-Vazirani, Simon's), see [Notebook 08b](08b-textbook-algorithms.ipynb).

In [1]:
import (
	"context"
	"fmt"
	"math"

	"github.com/janpfeifer/gonb/gonbui"
	"github.com/splch/goqu/algorithm/qpe"
	"github.com/splch/goqu/circuit/builder"
	"github.com/splch/goqu/circuit/draw"
	"github.com/splch/goqu/circuit/gate"
	"github.com/splch/goqu/sim/statevector"
	"github.com/splch/goqu/viz"
)

## Quantum Phase Estimation (QPE)

**Problem:** Given a unitary operator $U$ and one of its eigenstates
$|\psi\rangle$ satisfying $U|\psi\rangle = e^{2\pi i \varphi}|\psi\rangle$,
estimate the phase $\varphi$.

QPE is the most important subroutine in quantum computing. It underpins:
- **Shor's algorithm** (factoring via order-finding)
- **Quantum chemistry** (energy estimation of molecular Hamiltonians)
- **HHL algorithm** (linear systems solver)

**How it works:**

1. Prepare a **phase register** of $t$ qubits in superposition.
2. Apply controlled-$U^{2^k}$ operations to transfer phase information.
3. Apply the **inverse Quantum Fourier Transform** (QFT$^\dagger$) to convert
   phase into a binary fraction readable by measurement.

With $t$ phase bits, QPE estimates $\varphi$ to $t$ bits of precision:
$\hat{\varphi} = m / 2^t$ where $m$ is the measured integer.

For the **T gate** ($\pi/8$ gate), the eigenvalue on $|1\rangle$ is
$e^{i\pi/4} = e^{2\pi i \cdot 1/8}$, so the phase is $\varphi = 1/8 = 0.125$.

In [2]:
%%
ctx := context.Background()

// QPE on the T gate: T|1> = exp(i*pi/4)|1> = exp(2*pi*i * 1/8)|1>
// So the phase is 1/8 = 0.125
eigenCircuit, err := builder.New("eigen", 1).X(0).Build()
if err != nil {
	panic(err)
}

result, err := qpe.Run(ctx, qpe.Config{
	Unitary:      gate.T,
	NumPhaseBits: 3,
	EigenState:   eigenCircuit,
	Shots:        1000,
})
if err != nil {
	panic(err)
}

fmt.Println("=== Quantum Phase Estimation ===")
fmt.Printf("Unitary: T gate (phase = pi/4)\n")
fmt.Printf("Phase bits: %d\n", result.PhaseRegBits)
fmt.Printf("Estimated phase: %.4f (expected: 0.125)\n", result.Phase)
fmt.Printf("Phase as fraction: 1/8 = %.4f\n", 1.0/8.0)
fmt.Println("Measurement counts:", result.Counts)

=== Quantum Phase Estimation ===
Unitary: T gate (phase = pi/4)
Phase bits: 3
Estimated phase: 0.1250 (expected: 0.125)
Phase as fraction: 1/8 = 0.1250
Measurement counts: map[1100:1000]


In [3]:
%%
ctx := context.Background()

eigenCircuit, _ := builder.New("eigen", 1).X(0).Build()

result, _ := qpe.Run(ctx, qpe.Config{
	Unitary:      gate.T,
	NumPhaseBits: 3,
	EigenState:   eigenCircuit,
	Shots:        1000,
})

fmt.Println("QPE Circuit:")
gonbui.DisplayHTML(draw.SVG(result.Circuit))

svg := viz.Histogram(result.Counts, viz.WithTitle("QPE: T Gate (expected phase = 0.125)"))
gonbui.DisplayHTML(svg)

QPE Circuit:


q0 
 
 q1 
 
 q2 
 
 q3 
 
 X 
 H 
 H 
 H 
 
 
 T 
 
 
 T 
 
 
 T 
 
 
 T 
 
 
 T 
 
 
 T 
 
 
 T 
 
 
 
 H 
 
 
 P(-pi/2) 
 H 
 
 
 P(-pi/4) 
 
 
 P(-pi/2) 
 H 
 M 
 M 
 M

QPE: T Gate (expected phase = 0.125) 
 
 
 
 0 
 
 200 
 
 400 
 
 600 
 
 800 
 
 1000 
 
 1100

## State Preparation

Many quantum algorithms require initializing qubits in a specific state
beyond the standard $|0\rangle$. The `gate.MustStatePrep` function creates
a gate that prepares any normalized state from a list of amplitudes.

Internally, it uses a Mottonen decomposition -- a sequence of uniformly
controlled RY and RZ rotations that systematically constructs the target
state from $|0\rangle$.

Let's prepare the state $\frac{1}{2}|0\rangle + \frac{\sqrt{3}}{2}|1\rangle$,
which has $P(0) = 1/4$ and $P(1) = 3/4$.

In [4]:
%%
// Prepare a known state using gate.MustStatePrep
// Target: (1/2)|0> + (sqrt(3)/2)|1>  =>  P(0)=0.25, P(1)=0.75
amplitudes := []complex128{
	complex(0.5, 0),             // amplitude for |0>
	complex(math.Sqrt(3)/2, 0),  // amplitude for |1>
}

spGate := gate.MustStatePrep(amplitudes)

c, err := builder.New("state-prep", 1).
	Apply(spGate, 0).
	MeasureAll().
	Build()
if err != nil {
	panic(err)
}

sim := statevector.New(1)
counts, err := sim.Run(c, 2000)
if err != nil {
	panic(err)
}

fmt.Println("State preparation: (1/2)|0> + (sqrt(3)/2)|1>")
fmt.Println("Expected: P(0) = 0.25, P(1) = 0.75")
fmt.Println("Measured counts (2000 shots):", counts)
for outcome, count := range counts {
	fmt.Printf("  |%s> : %d (%.1f%%)\n", outcome, count, float64(count)/20.0)
}

svg := viz.Histogram(counts, viz.WithTitle("State Preparation: P(0)=0.25, P(1)=0.75"))
gonbui.DisplayHTML(svg)

State preparation: (1/2)|0> + (sqrt(3)/2)|1>
Expected: P(0) = 0.25, P(1) = 0.75
Measured counts (2000 shots): map[0:463 1:1537]
  |1> : 1537 (76.8%)
  |0> : 463 (23.1%)


State Preparation: P(0)=0.25, P(1)=0.75 
 
 
 
 0 
 
 500 
 
 1000 
 
 1500 
 
 2000 
 
 0 
 
 1

## Predict, Then Verify

**Question:** The S gate has the property $S|1\rangle = e^{i\pi/2}|1\rangle$. If we run QPE with 3 phase bits on the S gate, what phase $\varphi$ will it extract?

**Pause and predict** before running the next cell.

*Hint: Express the eigenvalue as $e^{2\pi i \varphi}$ and solve for $\varphi$. Is this phase exactly representable in 3 binary digits?*

In [5]:
%%
ctx := context.Background()

// Verify: QPE on the S gate should give phase = 0.25
eigenCircuit, _ := builder.New("eigen", 1).X(0).Build()

result, err := qpe.Run(ctx, qpe.Config{
	Unitary:      gate.S,
	NumPhaseBits: 3,
	EigenState:   eigenCircuit,
	Shots:        1000,
})
if err != nil {
	panic(err)
}

fmt.Printf("S gate -- Estimated phase: %.4f (expected: 0.2500)\n", result.Phase)
fmt.Printf("Match: %v\n", math.Abs(result.Phase-0.25) < 0.01)
fmt.Println("Counts:", result.Counts)
fmt.Println("\nPrediction confirmed: S gate phase = 1/4 = 0.25.")

S gate -- Estimated phase: 0.2500 (expected: 0.2500)
Match: true
Counts: map[1010:1000]

Prediction confirmed: S gate phase = 1/4 = 0.25.


## Self-Check Questions

Before attempting the exercises, make sure you can answer these:

1. How many queries does QPE need to estimate a phase to t bits of precision?
2. Why does QPE require the inverse QFT rather than the forward QFT?
3. What happens when the true phase is not exactly representable with the available number of phase bits?

---

## Exercises

### Exercise 1 -- QPE on a Custom Unitary

Run QPE on the **Z gate**. The Z gate applies a phase of $e^{i\pi}$ to
$|1\rangle$, which corresponds to $e^{2\pi i \cdot 1/2}$, giving phase
$\varphi = 1/2 = 0.5$. Verify that QPE recovers this phase with 3 phase bits.

In [6]:
%%
// Exercise 1: QPE on the Z gate
// Z|1> = -|1> = e^(i*pi)|1> = e^(2*pi*i * 1/2)|1>
// So the phase should be 0.5.
//
// TODO: Run QPE on the Z gate with:
//   - Unitary: gate.Z
//   - NumPhaseBits: 3
//   - EigenState: a circuit that prepares |1> (apply X to qubit 0)
//   - Shots: 1000
// Then verify that result.Phase is approximately 0.5.
//
// Hint: Use qpe.Run(ctx, qpe.Config{...})

// ctx := context.Background()
//
// eigenCircuit, _ := builder.New("eigen", 1).X(0).Build()
//
// result, err := qpe.Run(ctx, qpe.Config{
// 	Unitary:      gate.Z,
// 	NumPhaseBits: 3,
// 	EigenState:   eigenCircuit,
// 	Shots:        1000,
// })
// if err != nil {
// 	panic(err)
// }
//
// fmt.Printf("Z gate -- Estimated phase: %.4f (expected: 0.5000)\n", result.Phase)
// fmt.Printf("Match: %v\n", math.Abs(result.Phase-0.5) < 0.01)
// fmt.Println("Counts:", result.Counts)
//
// svg := viz.Histogram(result.Counts, viz.WithTitle("Exercise 1: QPE on Z Gate"))
// gonbui.DisplayHTML(svg)

fmt.Println("TODO: Uncomment the code above and fill in the missing parts!")

TODO: Uncomment the code above and fill in the missing parts!


### Exercise 2 -- QPE with Higher Precision

Run QPE on the T gate with **5 phase bits** instead of 3. Because the true
phase $1/8 = 0.00100$ in binary is exactly representable with 3 bits,
increasing to 5 bits should still produce a sharp peak at the same value.
Compare the measurement distributions.

In [7]:
%%
// Exercise 2: QPE on T gate with higher precision (5 phase bits)
//
// TODO: Run QPE on the T gate with both 3 and 5 phase bits,
// then compare the results.
//
// Steps:
//   1. Create an eigenstate circuit: builder.New("eigen", 1).X(0).Build()
//   2. Run qpe.Run with NumPhaseBits: 3
//   3. Run qpe.Run with NumPhaseBits: 5
//   4. Print both phases and compare with expected value 0.125
//
// Hint: The T gate phase 1/8 is exactly representable in 3 bits,
// so both should give the same sharp peak.

// ctx := context.Background()
//
// eigenCircuit, _ := builder.New("eigen", 1).X(0).Build()
//
// // TODO: Run QPE with 3 phase bits
// // result3, _ := qpe.Run(ctx, qpe.Config{
// // 	Unitary:      gate.T,
// // 	NumPhaseBits: 3,
// // 	EigenState:   eigenCircuit,
// // 	Shots:        1000,
// // })
//
// // TODO: Run QPE with 5 phase bits
// // result5, _ := qpe.Run(ctx, qpe.Config{
// // 	Unitary:      gate.T,
// // 	NumPhaseBits: 5,
// // 	EigenState:   eigenCircuit,
// // 	Shots:        1000,
// // })
//
// // fmt.Printf("3 phase bits -- phase: %.6f\n", result3.Phase)
// // fmt.Printf("5 phase bits -- phase: %.6f\n", result5.Phase)
// // fmt.Printf("Expected:               0.125000\n")
//
// // svg3 := viz.Histogram(result3.Counts, viz.WithTitle("QPE T Gate: 3 Phase Bits"))
// // svg5 := viz.Histogram(result5.Counts, viz.WithTitle("QPE T Gate: 5 Phase Bits"))
// // gonbui.DisplayHTML(svg3)
// // gonbui.DisplayHTML(svg5)

fmt.Println("TODO: Uncomment the code above and fill in the missing parts!")

TODO: Uncomment the code above and fill in the missing parts!


## Key Takeaways

1. **Quantum Phase Estimation** extracts the eigenvalue phase of a unitary
   operator using the inverse QFT. With $t$ ancilla qubits, it achieves
   $t$ bits of precision. QPE is the key subroutine in Shor's algorithm,
   quantum chemistry, and the HHL algorithm.

2. **State preparation** via `gate.MustStatePrep` uses Mottonen decomposition
   to initialize arbitrary quantum states from amplitude vectors.

3. The **controlled-$U^{2^k}$** gates transfer phase information from the
   unitary's eigenvalues into the phase register, where the inverse QFT
   converts it to a readable binary fraction.

4. When the true phase is exactly representable with $t$ bits, QPE recovers
   it perfectly. Otherwise, the result is a distribution peaked near the
   true phase, with precision improving exponentially with more phase bits.

For textbook oracle algorithms (Deutsch-Jozsa, Bernstein-Vazirani, Simon's), see [Notebook 08b](08b-textbook-algorithms.ipynb).

---

**Next up:** Notebook 09 -- Grover's Search Algorithm, where we'll harness amplitude amplification for unstructured search.